# 4. Model Training & Evaluation

Trains and compares Linear Regression, Random Forest, and Gradient Boosting; tunes the best model; saves the final pipeline. Run after `02_feature_engineering.ipynb`.

In [2]:
from pathlib import Path

def find_project_root(marker="requirements.txt", max_search_depth=4):
    """Locates the housing_project root so paths work no matter where
    Jupyter was launched from. Two strategies, tried in order:
    1. Walk UPWARD from the current directory (covers launching Jupyter
       from inside the project, e.g. from notebooks/pipeline/).
    2. Search DOWNWARD into subfolders (covers the common case of
       launching Jupyter from your home folder or Desktop, then browsing
       into the project through the Jupyter file browser -- the kernel's
       working directory stays at the launch folder, not the notebook's).
    """
    start = Path.cwd().resolve()

    # Strategy 1: search upward
    for parent in [start] + list(start.parents):
        if (parent / marker).exists():
            return parent

    # Strategy 2: search downward (breadth-first, limited depth)
    frontier = [start]
    for _ in range(max_search_depth):
        next_frontier = []
        for folder in frontier:
            try:
                subdirs = [d for d in folder.iterdir() if d.is_dir() and not d.name.startswith(".")]
            except PermissionError:
                continue
            for d in subdirs:
                if (d / marker).exists():
                    return d
                next_frontier.append(d)
        frontier = next_frontier
        if not frontier:
            break

    raise FileNotFoundError(
        f"Could not locate the project root (looking for '{marker}') starting from {start}.\n"
        f"Fix: either launch Jupyter from inside the housing_project folder, "
        f"or set PROJECT_ROOT manually below, e.g.:\n"
        f'    PROJECT_ROOT = Path(r"C:\\path\\to\\housing_project")'
    )

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")

Project root: D:\Projectsinterns\housing_project


In [3]:
import pandas as pd
import numpy as np
import joblib
import json
import time

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

FEATURES_PATH = PROJECT_ROOT / "data" / "housing_features.csv"
MODELS_DIR = PROJECT_ROOT / "models"
FIG_DIR = PROJECT_ROOT / "figures"
TARGET = "median_house_value"
DROP_COLS = ["median_house_value", "log_median_house_value"]

## Load features and split

In [4]:
df = pd.read_csv(FEATURES_PATH)
X = df.drop(columns=DROP_COLS)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(16512, 16) (4128, 16)


## Evaluation helper

In [5]:
def evaluate(model, X_test, y_test, name):
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    print(f"{name:22s} | RMSE: {rmse:10,.0f} | MAE: {mae:10,.0f} | R2: {r2:.4f}")
    return {"model": name, "rmse": rmse, "mae": mae, "r2": r2}

## Train and compare baseline models

In [6]:
models = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "RandomForest": RandomForestRegressor(
        n_estimators=150, max_depth=None, random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}
if HAS_XGB:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1
    )

results = []
fitted_models = {}
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    fitted_models[name] = model
    res = evaluate(model, X_test, y_test, name)
    res["train_time_sec"] = round(time.time() - t0, 2)
    results.append(res)

results_df = pd.DataFrame(results).sort_values("rmse")
results_df

LinearRegression       | RMSE:     72,563 | MAE:     50,866 | R2: 0.5982
RandomForest           | RMSE:     50,122 | MAE:     32,223 | R2: 0.8083
GradientBoosting       | RMSE:     53,510 | MAE:     36,486 | R2: 0.7815


,model,rmse,mae,r2,train_time_sec
1,RandomForest,50122.361584,32223.453884,0.808285,6.10
2,GradientBoosting,53510.415851,36486.160479,0.781491,5.26
0,LinearRegression,72562.864421,50866.209326,0.598189,0.08


## Hyperparameter tuning on the best model

Searched on a training subsample for speed, then refit on the full training set.

**Note:** `max_depth` is capped and `min_samples_leaf >= 2` is enforced for Random Forest — an uncapped tree (`max_depth=None`, `min_samples_leaf=1`) fully memorizes the data and can bloat the saved model to 200MB+, which is too large for a normal GitHub push.

In [7]:
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
print(f"Best baseline model: {best_name}")

if best_name in ("RandomForest", "GradientBoosting", "XGBoost"):
    search_sample_n = min(6000, len(X_train))
    X_search = X_train.sample(n=search_sample_n, random_state=42)
    y_search = y_train.loc[X_search.index]

    if best_name == "RandomForest":
        param_dist = {
            "n_estimators": [100, 150, 200],
            "max_depth": [15, 20, 25],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [2, 4, 6],
        }
        base = RandomForestRegressor(random_state=42, n_jobs=-1)
    elif best_name == "GradientBoosting":
        param_dist = {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [2, 3, 4],
            "subsample": [0.8, 1.0],
        }
        base = GradientBoostingRegressor(random_state=42)
    else:
        param_dist = {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [3, 4, 5],
            "subsample": [0.8, 1.0],
        }
        base = XGBRegressor(random_state=42, n_jobs=-1)

    search = RandomizedSearchCV(
        base, param_distributions=param_dist, n_iter=8,
        cv=3, scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1
    )
    search.fit(X_search, y_search)
    print(f"Best params (found on subsample): {search.best_params_}")

    best_model = base.__class__(**{**base.get_params(), **search.best_params_})
    best_model.fit(X_train, y_train)
    tuned_res = evaluate(best_model, X_test, y_test, f"{best_name} (tuned)")
    results.append(tuned_res)

Best baseline model: RandomForest
Best params (found on subsample): {'n_estimators': 150, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 20}
RandomForest (tuned)   | RMSE:     49,758 | MAE:     32,105 | R2: 0.8111


## Feature importance

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    print(importances.head(10))

    plt.figure(figsize=(8, 6))
    importances.head(10).sort_values().plot(kind="barh")
    plt.title(f"Top 10 Feature Importances ({best_name})")
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/06_feature_importance.png", dpi=120)
    plt.show()

median_income               0.490782
ocean_proximity_INLAND      0.141784
population_per_household    0.121295
longitude                   0.057078
latitude                    0.055462
housing_median_age          0.043665
rooms_per_household         0.023885
bedrooms_per_room           0.021957
total_rooms                 0.011046
total_bedrooms              0.010390
dtype: float64


C:\Users\ibrahim laptops\AppData\Local\Temp\ipykernel_21704\226910646.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save the final model

In [9]:
joblib.dump(best_model, f"{MODELS_DIR}/best_model.pkl", compress=3)
joblib.dump(list(X.columns), f"{MODELS_DIR}/feature_columns.pkl")

with open(f"{MODELS_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved best_model.pkl, feature_columns.pkl, and results.json")

Saved best_model.pkl, feature_columns.pkl, and results.json
